In [15]:
import mne
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from glob import glob
import scipy.io
import h5py
import os
from tqdm import tqdm

## Concatenate TF

In [147]:
csv_path = f"derivatives/behavior/"
output_path = f"derivatives/preprocessed/TF_arrays/"

# take IDs from fully processed behavioral data (checked for accuracy, validRT, missed responses) separately for each condition
sub_nonsoc = list(pd.read_csv(glob(f"{csv_path}thrive_data_nonsoc.csv")[0])["sub"])
sub_soc = list(pd.read_csv(glob(f"{csv_path}thrive_data_soc.csv")[0])["sub"])

# tf_files = sorted(glob(f"{data_path}/sub-*{condition}*.mat"))
for measure in [
    "TF",
    "ITPS",
    "ICPS",
    "wPLI"
]:
    if measure == "ITPS" or measure == "ICPS":
        key_idx = 1
    else:
        key_idx = -1
    data_path = f"derivatives/preprocessed/TF_outputs/main/resp/{measure}/"
    for condition in tqdm(["resp_ns_c_1", "resp_ns_i_0", "resp_ns_i_1",
                           "resp_s_i_1", "resp_s_c_1", "resp_s_i_0"]):
        if condition.split("_")[1] == "s":
            valid_sub_list = sub_soc.copy()
        elif condition.split("_")[1] == "ns":
            valid_sub_list = sub_nonsoc.copy()
        
        arr_list = []
        subjects_with_data = []    
        for sub_id in valid_sub_list:
            try:
                tf_files = sorted(glob(f"{data_path}/sub-{sub_id}*{measure}*{condition}*.mat"))
                # if len(tf_files) == 0:
                    # print(f"{sub_id} not in TF")
                data_file = h5py.File(tf_files[0])
                key_list = list(data_file.keys())
                data = data_file[key_list[key_idx]]
                assert data.shape == (64, 375, 59), "Check your data!"
                arr_list.append(data)
                subjects_with_data.append(sub_id)
            except: continue
            
        full_data = np.stack(arr_list, axis=0)
        assert full_data.shape[0] == len(subjects_with_data), "Check your data!"
        len(arr_list)
        scipy.io.savemat(f"{output_path}/{measure}_{condition}.mat",
                         {
                             f"{measure}_{condition}": full_data,
                             f"subjects": subjects_with_data,
                         })

132

## Inspect number of events

In [ ]:
import time
sub_to_inspect = "192"
trial_data = dict({
        "sub": [],
        "s_resp_incon_error": [],
        "s_resp_incon_corr": [],
        "ns_resp_incon_error": [],
        "ns_resp_incon_corr": [],
        "s_stim_incon_corr": [],
        "s_stim_con_corr": [],
        "ns_stim_incon_corr": [],
        "ns_stim_con_corr": [],
})

dataset_path = "/home/data/NDClab/datasets/thrive-dataset/"

sub_ids = sorted([i.split("/")[-1] for i in glob(
        f"{dataset_path}derivatives/preprocessed/sub-*{sub_to_inspect}*")])

list_of_eeg_file = sorted(
    glob(
        f"{dataset_path}derivatives/preprocessed/*{sub_to_inspect}*/s1_r1/eeg/*all_eeg_processed_data*.set")
)

start = time.time()

for file_idx, filename in enumerate(list_of_eeg_file):
    sub_id = sub_ids[file_idx].split("-")[-1]
    trial_data["sub"].append(sub_id)
    EEG = scipy.io.loadmat(filename, squeeze_me=True, struct_as_record=False)["EEG"]
    EEG_mne = mne.io.read_epochs_eeglab(filename, verbose = 'ERROR',)
    
    events = EEG.event
    n_times = EEG.pnts
    sr = EEG.srate
    num_ch = EEG.nbchan

    drop_idx = []
    for i in range(len(events)):
        latency = eeg_point2lat(
            [events[i].latency],
            [events[i].epoch],
            sr,
            timewin = [EEG.xmin*1000, EEG.xmax*1000],
            timeunit = 1e-3,
             )
        if latency >= -.1 and latency <= .1:
            drop_idx.append(i)
    
    events = [ev for ev in events if list(events).index(ev) in drop_idx]
    print(f"sub-{sub_id}: {len(events)} good events were found!")
    
    trial_data["s_resp_incon_error"].append(len(
        [ev for ev in events if\
        (ev.observation == "s") & (ev.eventType == "resp") & (ev.congruency == "i")\
        & (ev.accuracy == 0) & (ev.responded == 1) & (ev.validRt == 1) & (ev.extraResponse == 0)
    ]
    ))
    
    trial_data["s_resp_incon_corr"].append(len(
        [ev for ev in events if\
        (ev.observation == "s") & (ev.eventType == "resp") & (ev.congruency == "i")\
        & (ev.accuracy == 1) & (ev.responded == 1) & (ev.validRt == 1) & (ev.extraResponse == 0)
    ]
    ))
    
    trial_data["ns_resp_incon_error"].append(len(
        [ev for ev in events if\
        (ev.observation == "ns") & (ev.eventType == "resp") & (ev.congruency == "i")\
        & (ev.accuracy == 0) & (ev.responded == 1) & (ev.validRt == 1) & (ev.extraResponse == 0)
    ]
    ))
    
    trial_data["ns_resp_incon_corr"].append(len(
        [ev for ev in events if\
        (ev.observation == "ns") & (ev.eventType == "resp") & (ev.congruency == "i")\
        & (ev.accuracy == 1) & (ev.responded == 1) & (ev.validRt == 1) & (ev.extraResponse == 0)
    ]
    ))
    
    trial_data["s_stim_incon_corr"].append(len(
        [ev for ev in events if\
        (ev.observation == "s") & (ev.eventType == "stim") & (ev.congruency == "i")\
        & (ev.accuracy == 1) & (ev.responded == 1) & (ev.validRt == 1) & (ev.extraResponse == 0)
    ]
    ))
    
    trial_data["s_stim_con_corr"].append(len(
        [ev for ev in events if\
        (ev.observation == "s") & (ev.eventType == "stim") & (ev.congruency == "c")\
        & (ev.accuracy == 1) & (ev.responded == 1) & (ev.validRt == 1) & (ev.extraResponse == 0)
    ]
    ))
    
    trial_data["ns_stim_incon_corr"].append(len(
        [ev for ev in events if\
        (ev.observation == "ns") & (ev.eventType == "stim") & (ev.congruency == "i")\
        & (ev.accuracy == 1) & (ev.responded == 1) & (ev.validRt == 1) & (ev.extraResponse == 0)
    ]
    ))
    
    trial_data["ns_stim_con_corr"].append(len(
        [ev for ev in events if\
        (ev.observation == "ns") & (ev.eventType == "stim") & (ev.congruency == "c")\
        & (ev.accuracy == 1) & (ev.responded == 1) & (ev.validRt == 1) & (ev.extraResponse == 0)
    ]
    ))

end = time.time()
print(f"Executed time {np.round(end - start, 2)} s")

pd.DataFrame(trial_data)